# Shot Segregation — Google Colab

**What this does:** Takes animated videos (MP4) and produces a structured dataset of keyframe pairs + inbetween frames, suitable for training an inbetween frame generation GAN.

**Pipeline:**
1. Detect shot/scene boundaries (PySceneDetect)
2. Extract frames per shot (OpenCV)
3. Detect keyframes via SSIM-based visual difference analysis
4. Classify shots (character / background / text / blank)
5. Export structured PNG dataset

**Prerequisites:**
- Upload your video files to Google Drive under `MyDrive/InBetweenGenerator/InputVideos/`
- Upload the source code (`main.py`, `src/` folder) to `MyDrive/InBetweenGenerator/`

**Expected Drive structure:**
```
MyDrive/InBetweenGenerator/
├── main.py
├── src/
│   ├── __init__.py
│   ├── shot_detector.py
│   ├── frame_extractor.py
│   ├── keyframe_analyzer.py
│   ├── shot_classifier.py
│   ├── dataset_writer.py
│   └── validate_dataset.py
└── InputVideos/
    ├── video1.mp4
    └── ...
```

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install dependencies

In [ ]:
!pip install -q scenedetect[opencv] scikit-image tqdm

## 3. Setup paths

In [ ]:
import os
import sys

PROJECT_DIR = '/content/drive/MyDrive/InBetweenGenerator'
INPUT_VIDEOS_DIR = os.path.join(PROJECT_DIR, 'InputVideos')
OUTPUT_DIR = os.path.join(PROJECT_DIR, 'dataset_output')

# Add project dir to Python path so main.py can import from src/
sys.path.insert(0, PROJECT_DIR)

# Verify project files exist
assert os.path.exists(os.path.join(PROJECT_DIR, 'main.py')), \
    f'main.py not found in {PROJECT_DIR}. Upload the source code first.'
assert os.path.exists(os.path.join(PROJECT_DIR, 'src', '__init__.py')), \
    f'src/ package not found in {PROJECT_DIR}. Upload the src/ folder.'
assert os.path.isdir(INPUT_VIDEOS_DIR), \
    f'InputVideos/ not found at {INPUT_VIDEOS_DIR}. Create it and add your videos.'

# List available videos
video_exts = {'.mp4', '.avi', '.mkv', '.mov', '.webm'}
videos = [f for f in os.listdir(INPUT_VIDEOS_DIR)
          if os.path.splitext(f)[1].lower() in video_exts]
print(f'Project dir: {PROJECT_DIR}')
print(f'Input dir:   {INPUT_VIDEOS_DIR}')
print(f'Output dir:  {OUTPUT_DIR}')
print(f'\nFound {len(videos)} video(s):')
for v in sorted(videos):
    size_mb = os.path.getsize(os.path.join(INPUT_VIDEOS_DIR, v)) / (1024 * 1024)
    print(f'  - {v} ({size_mb:.1f} MB)')

## 4. Configure parameters

Adjust these thresholds for your content:

| Parameter | What it does | Range | Default |
|---|---|---|---|
| `SHOT_THRESHOLD` | Shot boundary sensitivity. Lower = more shots | 15-40 | 27.0 |
| `KEYFRAME_THRESHOLD` | SSIM threshold for keyframe detection. Lower = only large changes count | 0.7-0.95 | 0.85 |
| `MIN_INBETWEENS` | Minimum inbetween frames per segment | 1-10 | 1 |
| `MIN_SCENE_LEN` | Minimum frames per detected shot | 3-30 | 6 |
| `CHARACTERS_ONLY` | Filter out blank/text/background-only shots | True/False | False |

In [ ]:
# ----- Tunable parameters -----
SHOT_THRESHOLD = 27.0
KEYFRAME_THRESHOLD = 0.85
MIN_INBETWEENS = 1
MIN_SCENE_LEN = 6
CHARACTERS_ONLY = False
CHARACTER_THRESHOLD = 0.35

## 5. Copy videos to local SSD (important for speed)

Google Drive has very slow sequential I/O. Copying the video to Colab's local disk makes processing **~10x faster**. A 172K-frame video can take 30+ min from Drive but only ~3 min from local SSD.

In [ ]:
import shutil

# Copy videos to local SSD for fast I/O
LOCAL_VIDEOS = "/content/local_videos"
os.makedirs(LOCAL_VIDEOS, exist_ok=True)

for v in sorted(videos):
    src = os.path.join(INPUT_VIDEOS_DIR, v)
    dst = os.path.join(LOCAL_VIDEOS, v)
    if not os.path.exists(dst):
        size_mb = os.path.getsize(src) / (1024 * 1024)
        print(f"Copying {v} ({size_mb:.0f} MB) to local SSD...")
        shutil.copy2(src, dst)
        print(f"  Done!")
    else:
        print(f"{v}: already copied locally")

# Use local path from now on
INPUT_VIDEOS_DIR = LOCAL_VIDEOS
print(f"\nWill process from: {INPUT_VIDEOS_DIR}")


## 6. Run shot segregation

Processes all videos in `InputVideos/` and writes the dataset to `dataset_output/`.

In [ ]:
import logging
from pathlib import Path

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='[%(levelname)s] %(name)s: %(message)s',
    force=True,
)

# Import from the project
from src.shot_detector import detect_shots
from src.frame_extractor import extract_frames
from src.keyframe_analyzer import detect_keyframes
from src.dataset_writer import write_shot, write_dataset_summary
from src.shot_classifier import classify_shot, ShotType

from tqdm.notebook import tqdm

input_path = Path(INPUT_VIDEOS_DIR)
output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)

video_files = sorted(
    p for p in input_path.iterdir()
    if p.is_file() and p.suffix.lower() in video_exts
)

print(f'Processing {len(video_files)} video(s)...\n')

for video_path in video_files:
    video_name = video_path.stem
    video_output_dir = output_path / video_name
    video_output_dir.mkdir(parents=True, exist_ok=True)

    print('=' * 60)
    print(f'Processing: {video_path.name}')
    print('=' * 60)

    # Step 1: Detect shots
    shots, fps, total_frames = detect_shots(
        str(video_path),
        threshold=SHOT_THRESHOLD,
        min_scene_len=MIN_SCENE_LEN,
    )
    print(f'Found {len(shots)} shot(s) in {total_frames} frames ({fps:.2f} fps)')

    if not shots:
        print('No shots detected - skipping.')
        continue

    # Step 2-3: Process each shot
    all_shot_metadata = []

    for shot_idx, (start_frame, end_frame) in enumerate(
        tqdm(shots, desc=f'Shots ({video_name})', unit='shot'), start=1
    ):
        frames = extract_frames(str(video_path), start_frame, end_frame)

        if len(frames) < 3:
            continue

        # Optional character filtering
        if CHARACTERS_ONLY:
            shot_type, confidence = classify_shot(
                frames, character_threshold=CHARACTER_THRESHOLD
            )
            if shot_type != ShotType.CHARACTER:
                continue

        # Detect keyframes
        segments = detect_keyframes(
            frames,
            keyframe_threshold=KEYFRAME_THRESHOLD,
            min_inbetweens=MIN_INBETWEENS,
        )

        if not segments:
            continue

        # Write to disk
        shot_meta = write_shot(
            shot_index=shot_idx,
            segments=segments,
            shot_start_frame=start_frame,
            shot_end_frame=end_frame,
            fps=fps,
            video_output_dir=video_output_dir,
        )
        all_shot_metadata.append(shot_meta)

    # Write summary
    if all_shot_metadata:
        write_dataset_summary(
            video_name=video_name,
            shot_metadata=all_shot_metadata,
            fps=fps,
            total_video_frames=total_frames,
            video_output_dir=video_output_dir,
        )
        print(f'\nDone: {video_name} -> {len(all_shot_metadata)} shots written to {video_output_dir}')
    else:
        print(f'\nNo valid segments produced for {video_name}.')

print(f'\n{"=" * 60}')
print(f'All videos processed. Dataset at: {output_path}')

## 7. Validate dataset

In [ ]:
from src.validate_dataset import validate_video_dataset

output_path = Path(OUTPUT_DIR)
total_errors = 0

for video_dir in sorted(output_path.iterdir()):
    if not video_dir.is_dir():
        continue
    errors = validate_video_dataset(video_dir)
    if errors:
        print(f'\n{video_dir.name}: {len(errors)} error(s)')
        for e in errors:
            print(f'  - {e}')
        total_errors += len(errors)
    else:
        print(f'{video_dir.name}: OK')

if total_errors == 0:
    print('\nAll datasets valid!')
else:
    print(f'\nTotal errors: {total_errors}')

## 8. Dataset statistics

In [ ]:
import json

output_path = Path(OUTPUT_DIR)
total_shots = 0
total_segments = 0
total_inbetweens = 0

for video_dir in sorted(output_path.iterdir()):
    if not video_dir.is_dir():
        continue

    summary_path = video_dir / 'dataset_summary.json'
    if summary_path.exists():
        with open(summary_path) as f:
            summary = json.load(f)
        v_shots = summary.get('num_shots', 0)
        v_segments = summary.get('total_segments', 0)
        v_inbetweens = summary.get('total_inbetween_frames', 0)
        print(f'{video_dir.name}: {v_shots} shots, {v_segments} segments, {v_inbetweens} inbetween frames')
        total_shots += v_shots
        total_segments += v_segments
        total_inbetweens += v_inbetweens
    else:
        # Count manually
        shot_dirs = [d for d in video_dir.iterdir() if d.is_dir() and d.name.startswith('shot_')]
        v_shots = len(shot_dirs)
        v_segments = 0
        v_inbetweens = 0
        for sd in shot_dirs:
            for seg in sd.iterdir():
                if seg.is_dir() and seg.name.startswith('segment_'):
                    v_segments += 1
                    ib_dir = seg / 'inbetweens'
                    if ib_dir.exists():
                        v_inbetweens += len(list(ib_dir.glob('*.png')))
        print(f'{video_dir.name}: {v_shots} shots, {v_segments} segments, {v_inbetweens} inbetween frames')
        total_shots += v_shots
        total_segments += v_segments
        total_inbetweens += v_inbetweens

print(f'\n--- Totals ---')
print(f'Shots:     {total_shots}')
print(f'Segments:  {total_segments}')
print(f'Inbetweens (training samples): {total_inbetweens}')

## 9. Preview sample segments

Displays keyframe pairs and a few inbetween frames from random segments.

In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt

output_path = Path(OUTPUT_DIR)

# Collect all segment directories
all_segments = []
for video_dir in sorted(output_path.iterdir()):
    if not video_dir.is_dir():
        continue
    for shot_dir in sorted(video_dir.iterdir()):
        if not shot_dir.is_dir() or not shot_dir.name.startswith('shot_'):
            continue
        for seg_dir in sorted(shot_dir.iterdir()):
            if seg_dir.is_dir() and seg_dir.name.startswith('segment_'):
                all_segments.append(seg_dir)

NUM_PREVIEW = min(3, len(all_segments))
preview_segments = random.sample(all_segments, NUM_PREVIEW) if all_segments else []

for seg_dir in preview_segments:
    key_first = seg_dir / 'key_first.png'
    key_last = seg_dir / 'key_last.png'
    inbetweens_dir = seg_dir / 'inbetweens'

    if not key_first.exists() or not key_last.exists():
        continue

    inbetween_files = sorted(inbetweens_dir.glob('*.png')) if inbetweens_dir.exists() else []

    # Pick up to 3 evenly-spaced inbetweens
    if len(inbetween_files) > 3:
        indices = [0, len(inbetween_files) // 2, len(inbetween_files) - 1]
        selected_ib = [inbetween_files[i] for i in indices]
    else:
        selected_ib = inbetween_files

    images = [Image.open(key_first)] + [Image.open(p) for p in selected_ib] + [Image.open(key_last)]
    titles = ['Key First'] + [f'Inbetween {i+1}' for i in range(len(selected_ib))] + ['Key Last']

    fig, axes = plt.subplots(1, len(images), figsize=(4 * len(images), 4))
    if len(images) == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img)
        ax.set_title(title, fontsize=10)
        ax.axis('off')

    rel_path = seg_dir.relative_to(output_path)
    fig.suptitle(str(rel_path), fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

if not all_segments:
    print('No segments found. Run step 5 first.')

## 10. (Optional) Process a single video

If you want to test with just one video before running the full batch:

In [ ]:
# Change this to the filename of the specific video you want to process
# SINGLE_VIDEO = 'your_video.mp4'

# video_path = Path(INPUT_VIDEOS_DIR) / SINGLE_VIDEO
# assert video_path.exists(), f'Video not found: {video_path}'

# from main import process_video
# process_video(
#     video_path=video_path,
#     output_dir=Path(OUTPUT_DIR),
#     shot_threshold=SHOT_THRESHOLD,
#     keyframe_threshold=KEYFRAME_THRESHOLD,
#     min_inbetweens=MIN_INBETWEENS,
#     min_scene_len=MIN_SCENE_LEN,
#     characters_only=CHARACTERS_ONLY,
#     character_threshold=CHARACTER_THRESHOLD,
# )

print('Uncomment the code above and set SINGLE_VIDEO to use this cell.')

## 11. Check output disk usage

In [ ]:
import subprocess

output_path_str = str(Path(OUTPUT_DIR))
if os.path.exists(output_path_str):
    result = subprocess.run(
        ['du', '-sh', output_path_str],
        capture_output=True, text=True
    )
    print(f'Dataset size: {result.stdout.strip()}')

    # Per-video breakdown
    result = subprocess.run(
        ['du', '-sh'] + sorted(
            str(d) for d in Path(output_path_str).iterdir() if d.is_dir()
        ),
        capture_output=True, text=True
    )
    print(f'\nPer-video breakdown:')
    print(result.stdout)
else:
    print('Output directory does not exist yet. Run step 5 first.')